# 00 AI Infra 学习地图：从 Agent/RAG 开发者到 Infra 视角

![AI Infra stack](../assets/images/ai_infra_stack_cover_unlabeled.png)

这门教程面向的不是从零学习深度学习算法的人，也不是专门写 CUDA kernel 的系统工程师，而是已经会做 Agent、RAG、模型 API 接入和业务开发的人。你的下一步能力是：当一个 AI 功能要真正上线时，你知道它为什么慢、为什么贵、为什么不稳定、为什么难以观测，以及应该从哪一层改。

AI Infra 可以理解为“让模型能力变成可靠产品能力”的工程系统。单次 demo 只要能回答就行；生产系统要同时满足质量、延迟、吞吐、成本、安全、可观测、可回滚和可评估。这个差别就是本教程要补的核心。


## 1. 一次 AI 请求经过哪些层

如果一个企业知识库助手收到用户问题，表面上看只是“检索 + 大模型回答”。但上线后，真实链路通常更长：

```mermaid
flowchart TB
    U["用户 / 业务系统"] --> Auth["鉴权与限流"]
    Auth --> App["AI 应用编排<br/>Agent / RAG / Workflow"]
    App --> Retrieve["检索层<br/>query rewrite / hybrid search / rerank"]
    Retrieve --> Prompt["Prompt 组装<br/>系统提示 / 上下文 / 工具说明"]
    Prompt --> Gateway["模型网关<br/>路由 / fallback / 预算 / 审计"]
    Gateway --> Engine["推理引擎<br/>vLLM / SGLang / TRT-LLM / TGI"]
    Engine --> GPU["计算资源<br/>GPU / Metal / CPU / K8s"]
    Engine --> Stream["流式输出<br/>TTFT / ITL / tokens/sec"]
    Stream --> App
    App --> Logs["日志与 Trace"]
    App --> Eval["质量评测与反馈"]
    Logs --> Ops["告警 / 排障 / 回滚"]
    Eval --> Release["灰度 / 门禁 / 迭代"]
```

这张图里每一层都可能成为问题来源。回答慢可能是检索慢、排队慢、prefill 慢、decode 慢，也可能是网关 fallback 到了更慢的供应商。回答差可能是数据版本错、chunk 切分差、rerank 失败、模型变更、prompt 被污染，也可能是工具调用失败后没有降级。


## 2. AI Infra 的核心分层

| 层 | 你要理解什么 | 常见组件 | 典型问题 |
|---|---|---|---|
| 模型层 | 模型大小、上下文、量化、能力边界 | Qwen/Llama/DeepSeek/Embedding/Reranker | 选错模型、成本过高、能力不稳 |
| 推理层 | prefill、decode、KV cache、batching、调度 | vLLM/SGLang/TensorRT-LLM/TGI/MLX | TTFT 高、ITL 高、OOM、吞吐低 |
| 资源层 | GPU 显存、带宽、dtype、并行、容器 | CUDA/Metal/Kubernetes/GPU Operator | GPU 利用率低、资源碎片、扩容慢 |
| 网关层 | 统一 API、鉴权、限流、路由、fallback、预算 | LiteLLM/自研 Gateway/API Gateway | 调用散乱、无法控成本、故障扩散 |
| 数据层 | ingestion、chunk、embedding、vector DB、rerank | ETL/Vector DB/Object Store/Queue | 数据过期、召回差、版本不可追踪 |
| 观测层 | metrics、logs、traces、request id、token 账单 | Prometheus/Grafana/OTel/Langfuse | 慢在哪里不知道、质量退化不自知 |
| 评测层 | offline eval、canary、regression、人工反馈 | eval dataset/scorer/dashboard | 上线靠感觉、改动无质量门禁 |
| 安全治理 | prompt injection、PII、工具权限、审计 | policy/sandbox/human approval | 泄露数据、错误执行高风险工具 |

一个“懂 AI Infra 的产品/研发”不一定亲自维护所有组件，但要能把问题拆到正确层级，知道每层的关键指标和常见权衡。


## 3. 从 Agent/RAG 经验迁移到 Infra 视角

你已有 Agent/RAG 基础，这是很好的起点。区别在于，应用开发通常关注“功能能不能做出来”，Infra 视角关注“功能能不能稳定、可控、低成本地长期运行”。

| Agent/RAG 开发问题 | Infra 追问 |
|---|---|
| prompt 怎么写效果好？ | prompt 版本如何管理？变更如何回归评测？ |
| top-k 取多少？ | 召回率、精确率、rerank 延迟和 token 成本如何权衡？ |
| 调哪个模型？ | 这个模型的 TTFT、ITL、吞吐、并发、失败率和成本是多少？ |
| 工具调用怎么实现？ | 工具权限、超时、重试、幂等、人审、审计怎么做？ |
| 用户说回答错了怎么办？ | 能否从 trace 还原 retrieval、prompt、model output、tool call 和版本？ |

这套教程的学习顺序就是沿着这个迁移路径设计的：先补模型推理基础，再补 GPU/显存，再学推理服务，再学服务治理和 RAG/Agent 生产化。


In [ ]:
# 一个端到端请求的 latency breakdown。真实系统可从 trace/metrics 中采集这些分段。
from dataclasses import dataclass

@dataclass
class Stage:
    name: str
    ms: int
    layer: str

stages = [
    Stage('auth_rate_limit', 8, 'gateway'),
    Stage('query_rewrite', 25, 'app'),
    Stage('hybrid_search', 55, 'data'),
    Stage('rerank', 35, 'data'),
    Stage('prompt_assembly', 8, 'app'),
    Stage('model_queue', 40, 'serving'),
    Stage('prefill', 180, 'serving'),
    Stage('first_decode_step', 20, 'serving'),
    Stage('stream_decode', 520, 'serving'),
    Stage('trace_eval_sample', 10, 'observability'),
]

total = sum(s.ms for s in stages)
by_layer = {}
for s in stages:
    by_layer[s.layer] = by_layer.get(s.layer, 0) + s.ms

print('total latency:', total, 'ms')
for layer, ms in sorted(by_layer.items(), key=lambda x: x[1], reverse=True):
    print(f'{layer:14s} {ms:4d} ms {ms / total:5.1%}')


## 4. 企业 RAG 助手上线案例

假设你要上线一个“公司制度问答助手”。demo 阶段，你可能只需要：读取 PDF、切 chunk、存向量库、检索 top-k、拼 prompt、调用模型。但生产阶段至少要补这些东西：

1. **数据管道**：文档从哪里来，谁有权限，什么时候更新，解析失败怎么办，chunk 版本如何追踪。
2. **检索质量**：不仅看最终答案，还要看 top-k 是否包含正确依据、rerank 是否把关键段落排到前面。
3. **推理性能**：长制度文档会让 prompt 变长，prefill 成本上升，TTFT 变差；高并发会让 KV cache 和队列时间上升。
4. **模型网关**：不同部门可能有不同预算和权限，需要 virtual key、限流、模型路由和 fallback。
5. **观测与评测**：每次回答要能追踪 request id、检索段落、prompt 版本、模型版本、token 数、延迟、人工反馈。
6. **安全治理**：内部制度可能包含敏感内容，必须处理权限过滤、PII、prompt injection 和越权检索。

这就是 AI Infra 的意义：把“一个能跑的 AI demo”变成“一个能负责的 AI 系统”。


## 5. 学习检查清单

读完整个项目后，你应该能回答：

- 为什么 LLM 推理分 prefill 和 decode？
- 为什么长上下文会吃 KV cache，而不仅是计算慢？
- 为什么 GPU 显存不是只装模型权重？
- vLLM/SGLang/TensorRT-LLM/TGI/Ray Serve 分别适合什么场景？
- 一个服务的 TTFT、ITL、吞吐、queue time 分别说明什么？
- K8s 里 GPU 推理服务为什么需要 GPU Operator、resource request 和 autoscaling？
- 模型网关为什么要做 virtual key、fallback、budget 和 observability？
- RAG/Agent 从 demo 到 production，最容易漏掉哪些基础设施？


## 6. AI Infra 团队和业务团队如何协作

在公司里，AI Infra 往往不是一个单人职责，而是平台、算法、应用、数据、安全、SRE 多方协作。产品/研发要知道各方边界，这样排查和推进时不会只说“模型不行”。

| 角色 | 典型职责 | 你要如何协作 |
|---|---|---|
| 应用研发 | Agent/RAG 编排、业务逻辑、用户体验 | 提供真实 workload、失败样例、业务约束 |
| 算法/模型团队 | 模型选择、微调、评测、模型能力边界 | 明确任务指标、错误类别、模型升级风险 |
| Infra/平台团队 | serving、网关、GPU、K8s、观测、成本 | 提供流量模型、SLA、预算和上线窗口 |
| 数据团队 | 文档管道、数据权限、质量、更新 | 定义数据来源、版本、权限和更新频率 |
| 安全/合规 | 敏感数据、审计、权限、工具风险 | 明确哪些输入/输出/工具需要审计或人审 |
| SRE/运维 | 告警、事故响应、容量、稳定性 | 定义 SLO、告警阈值和回滚策略 |

一个成熟的 AI 功能评审，不只问“模型效果怎么样”，还会问：峰值流量是多少？p95 prompt 多长？能否追踪每次回答的证据和版本？如果模型服务不可用，业务怎么降级？如果用户输入 prompt injection，系统怎么限制工具和检索权限？


## 7. 三类学习深度：产品、研发、平台

同一套 AI Infra 知识，不同岗位需要的深度不同：

- **AI 产品**：需要会拆指标、判断取舍、定义 SLA、理解成本和风险。你不一定写 K8s YAML，但要知道长上下文为什么贵、fallback 为什么会导致预算暴涨、线上质量为什么要 eval gate。
- **AI 应用研发**：需要会接网关、写可观测链路、处理超时重试、设计 RAG/Agent 数据流。你不一定写 CUDA，但要知道 TTFT/ITL、token budget、KV cache、模型版本和 prompt 版本。
- **AI Infra/平台研发**：需要深入 serving engine、GPU、调度、Kubernetes、网关、多租户、监控和成本优化。

本项目默认目标是前两者向第三者靠近：你不一定成为平台工程师，但要能和平台工程师高质量沟通，并能做出合理产品/研发判断。


In [ ]:
# 用一个风险矩阵帮助产品/研发做上线前检查。
risks = [
    ('p95 prompt too long', 'latency/cost', 4, 4),
    ('missing retrieval permission filter', 'security', 5, 5),
    ('no request id in logs', 'operability', 3, 4),
    ('no offline regression set', 'quality', 4, 5),
    ('fallback to expensive cloud model', 'cost', 3, 5),
]

print('risk, category, severity, probability, priority')
for name, category, severity, prob in sorted(risks, key=lambda x: x[2]*x[3], reverse=True):
    print(f'{name:36s} {category:12s} {severity} {prob} {severity*prob:2d}')


## 8. 贯穿全教程的企业助手案例

后续章节都可以用同一个案例来理解：一家金融科技公司要做“内部投研制度与客户沟通合规助手”。用户可能问制度、查产品说明、总结会议纪要、生成客户邮件草稿。这个功能看似是 RAG + Agent，但真正上线时会牵涉完整 AI Infra。

第一阶段是 demo：把制度文档解析成 chunk，写入向量库，检索后拼 prompt，调用模型回答。这时你主要关注能不能答出来。

第二阶段是内测：用户变多，问题开始暴露。有人问的问题很长，TTFT 变慢；有人上传新制度后答案仍引用旧制度；某些回答没有引用来源；同一个问题不同时间答案不同；模型 API 账单突然升高。这时你开始需要 token 统计、文档版本、prompt 版本、模型版本、日志和评测集。

第三阶段是生产：系统接入公司账号和权限。不同部门只能看到自己的文档；客户邮件草稿必须经过合规检查；高风险工具调用要人审；模型服务不可用时要降级；每个部门有预算；每次回答都要能审计。这时你需要网关、权限过滤、observability、eval gate、灰度、回滚和事故流程。

这个案例说明：AI Infra 不是抽象概念，而是每个“从 demo 到 production”的缺口。


## 9. 自测题

1. 为什么 AI Infra 不能只等同于“部署一个模型 API”？
2. 一个 RAG 请求慢，至少应该拆成哪些阶段排查？
3. 为什么 Agent 工具调用比普通文本生成更需要安全治理？
4. 如果只能加三类观测字段，你会选 request id、token usage、latency breakdown、model version、prompt version 中的哪些？为什么？
5. 产品经理为什么也需要理解 TTFT、ITL 和 token 成本？

建议你带着这些问题读后续章节。每章学完后，都回到企业助手案例，问自己“这一章能帮我解决案例中的哪个生产问题”。


## 10. 如何把这套知识用于求职项目叙述

如果你在简历或面试中讲 Agent/RAG 项目，建议不要只讲“用了 LangChain、向量库和某个模型”。更有区分度的讲法是按 Infra 维度组织：

- **业务目标**：解决什么用户问题，为什么需要 LLM/RAG/Agent。
- **数据链路**：文档来源、解析、chunk、embedding、版本和权限。
- **推理链路**：模型选择、上下文长度、token 成本、流式输出、fallback。
- **服务治理**：网关、限流、鉴权、预算、request id、日志和 trace。
- **质量闭环**：golden set、检索指标、答案指标、线上反馈、回归测试。
- **稳定性**：超时、重试、降级、回滚、告警和事故复盘。

例如一句普通表述是：“我做了一个企业知识库 RAG 助手”。更成熟的表述是：“我做了企业知识库助手，并把链路拆成 ingestion、retrieval、rerank、prompt assembly、LLM serving 和 observability；对文档 chunk 做版本追踪，对模型调用记录 request id/token/latency，对常见问题维护 golden set，并用 TTFT、检索命中率、答案引用一致性和 token 成本做迭代。”后者体现的是 AI Infra 思维。


## 来源与延伸阅读

本章正文已经把关键内容消化进教程，下面链接只用于延伸阅读和核对最新细节：vLLM、vLLM-Metal、Ray Serve LLM、SGLang、TensorRT-LLM、Text Generation Inference、LiteLLM、KServe、NVIDIA GPU Operator、OpenTelemetry GenAI。
